# Pipeline evaluation — v2 (kernel-crash-safe + version verification)

Run this any time you change `build_benefits.py`, prompts, or section codes.

**Key safeguards:**
- Cell 1 verifies the server is running the expected `build_benefits` version. If it's not, the notebook STOPS immediately. No more wasted 11-minute test runs against stale code.
- Polling is rewritten to be crash-safe — no per-poll print spam, no growing output that overwhelms the kernel.
- Result fetching happens in chunks if needed.


## 0. Config

In [ ]:
import math
import json
import time
import re
import os
import sys
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
import requests
import pandas as pd

# ── Where to test ─────────────────────────────────────────────────────────────
BASE_URL = 'http://127.0.0.1:8080'

# ── Required server version ───────────────────────────────────────────────────
# If the server doesn't report this version, the notebook STOPS. This prevents
# the very common waste-of-time pattern: running tests against stale code.
EXPECTED_BUILD_VERSION = '2026-05-12-sweeper-and-retry-v5'

# ── Input + reference paths ───────────────────────────────────────────────────
PAYLOAD_PATH   = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')
REFERENCE_PATH = None   # set to a Path('./reference.json') when you have one

# ── Scoreboard ────────────────────────────────────────────────────────────────
SCOREBOARD_PATH = Path('./eval_scoreboard.csv')

# ── Run config ────────────────────────────────────────────────────────────────
POLL_INTERVAL  = 30      # poll every 30s instead of 10s — avoids overwhelming the kernel
MAX_WAIT       = 1800    # 30 min — for very large loads
POST_TIMEOUT   = 180
RUN_LABEL      = 'v4-HCSC-targets'

print(f'Will hit {BASE_URL}')
print(f'Expecting build_benefits version: {EXPECTED_BUILD_VERSION}')


## 1. Pre-flight check — verify server is running the right code

This is the single most important diagnostic. If you skip it, you risk wasting 10+ minutes on a test against stale code (which is what happened last time).

In [ ]:
r = requests.get(f'{BASE_URL}/', timeout=15)
health = r.json()
print(f'Server health: {r.status_code}')
print(f'  build_benefits_ver: {health.get("build_benefits_ver", "<not exposed>")}')
print(f'  timestamp:          {health.get("timestamp")}')

server_ver = health.get('build_benefits_ver')
if server_ver != EXPECTED_BUILD_VERSION:
    msg = (
        f'\n  STOPPING — server is running {server_ver!r}, expected {EXPECTED_BUILD_VERSION!r}.\n'
        f'  Update main.py + build_benefits.py on the server and restart Flask.\n'
        f'  Then re-run this notebook.'
    )
    raise SystemExit(msg)
else:
    print(f'\n✓ Server is running the expected version. Proceeding.')


## 2. POST the payload

Returns 202 immediately. Captures the load_id for polling.

In [ ]:
def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw
clean = sanitize(rows)

input_plans = {r['FileName'].strip() for r in clean if r.get('FileName')}
print(f'Input: {len(clean):,} PBP rows / {len(input_plans)} distinct plans')

t_post = time.monotonic()
r = requests.post(f'{BASE_URL}/save', json=clean,
                  headers={'Content-Type': 'application/json'}, timeout=POST_TIMEOUT)
post_time = time.monotonic() - t_post

if r.status_code != 202:
    raise RuntimeError(f'POST failed: {r.status_code} {r.text}')

resp = r.json()
load_id = resp['load_id']
print(f'\nPOST: {r.status_code} in {post_time:.1f}s')
print(f'  load_id:   {load_id}')
print(f'  plan_count from server: {resp.get("plan_count")}')


## 3. Poll until done — crash-safe version

Uses `clear_output` so the cell output doesn't grow unbounded. Polls every 30s instead of 10s. Watch the Flask terminal for the actual processing progress.

In [ ]:
from IPython.display import clear_output

elapsed = 0
t_proc = time.monotonic()
output_data = None
poll_count = 0

print(f'Polling /results/{load_id} every {POLL_INTERVAL}s (max {MAX_WAIT}s)...')
print('Watch the Flask terminal for actual per-benefit progress.\n')

while elapsed < MAX_WAIT:
    try:
        r = requests.get(f'{BASE_URL}/results/{load_id}', timeout=30)
        data = r.json()
        status = data.get('status')
        poll_count += 1

        # Refresh the cell output instead of appending — keeps kernel happy
        clear_output(wait=True)
        print(f'Polling load_id={load_id}')
        print(f'Elapsed:    {elapsed}s')
        print(f'Status:     {status}')
        print(f'Polls:      {poll_count}')
        print(f'Time/poll:  {POLL_INTERVAL}s')

        if status == 'success':
            output_data = data
            break
        if status == 'error':
            raise RuntimeError(f'Pipeline error: {data.get("error")}')
    except requests.exceptions.ReadTimeout:
        print(f'  (poll timeout at {elapsed}s, retrying)')

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

proc_time = time.monotonic() - t_proc
total_time = post_time + proc_time

if output_data is None:
    print(f'\n⚠ Timed out after {MAX_WAIT}s. Job may still be running.')
    print(f'  To fetch later: requests.get("{BASE_URL}/results/{load_id}").json()')
    raise SystemExit('Stopping — no results within MAX_WAIT')

output_rows = output_data['results']
print(f'\n✓ Done')
print(f'  Processing time: {proc_time:.1f}s')
print(f'  Total time:      {total_time:.1f}s')
print(f'  Plans:           {output_data.get("plan_count")}')
print(f'  Rows out:        {output_data.get("result_count")}')


## 4. Coverage analysis — what extracted vs what's missing

In [ ]:
BENEFIT_TARGETS = [
    ('610',  'Health Plan Deductible',     [],             ['Health Plan Deductible', 'Medical Deductible', 'Annual Plan Deductible']),
    ('611',  'Drug Deductible',            [],             ['Drug Deductible', 'Rx Deductible', 'Enter Deductible Amount']),
    ('615',  'Drug Monthly Premium',       [],             ['Drug Monthly Premium', 'Rx Premium', 'Part D Premium']),
    ('616',  'Part B Premium Reduction',   [],             ['Part B Premium', 'Part B Reduction', 'Part B giveback']),
    ('620',  'Out-of-Pocket Spending',     [],             ['MOOP', 'Max Enrollee Cost', 'Out of Pocket', 'OOP']),
    ('700',  'Tier Names',                 [],             ['Rx Tier', 'Formulary Tier', 'Tier Names']),
    ('710',  'Initial Coverage',           [],             ['Initial Coverage Phase', 'Rx Setup']),
    ('711',  'Retail Pharmacy',            [],             ['Retail Pharmacy', 'Initial Coverage Phase']),
    ('730',  'Catastrophic Coverage',      [],             ['Catastrophic Coverage']),
    ('740',  'Formulary Exception',        [],             ['Formulary Exception']),
    ('755',  'Preferred Mail Order',       [],             ['Preferred Mail Order']),
    ('760',  'Standard Mail Order',        [],             ['Standard Mail Order']),
    ('800',  'Inpatient Hospital',         ['1a'],         ['Inpatient Hospital']),
    ('810',  'Inpatient Mental Health',    ['1b'],         ['Inpatient Mental Health', 'Inpatient Psychiatric']),
    ('820',  'Skilled Nursing Facility',   ['2'],          ['Skilled Nursing', 'SNF']),
    ('900',  'PCP',                        ['7a'],         ['Primary Care', 'PCP']),
    ('910',  'Specialist',                 ['7b', '7d'],   ['Specialist', 'Specialty Care']),
    ('911',  'Telehealth',                 ['7d', '7j'],   ['Telehealth', 'Telemedicine', 'Virtual', 'Additional Telehealth']),
    ('920',  'Chiropractic',               ['8'],          ['Chiropractic']),
    ('930',  'Podiatry',                   ['9a'],         ['Podiatry']),
    ('940',  'Outpatient Mental Health',   ['4a'],         ['Outpatient Mental Health']),
    ('950',  'Outpatient Substance Abuse', ['4b'],         ['Substance Abuse']),
    ('960',  'Outpatient Surgery',         ['4c','9a1','9a2','9b','9d'], ['Outpatient Surgery', 'Outpatient Hospital', 'Ambulatory Surgical']),
    ('970',  'Ambulance',                  ['5','5a','5b'],['Ambulance']),
    ('981',  'Emergency',                  ['4a','6a'],    ['Emergency Services']),
    ('982',  'Urgent Care',                ['4b','6b'],    ['Urgent Care', 'Urgently Needed']),
    ('990',  'Outpatient Rehab',           ['10','5b'],    ['Outpatient Rehabilitation', 'Physical Therapy']),
    ('1000', 'DME',                        ['11a'],        ['Durable Medical Equipment', 'DME']),
    ('1020', 'Diabetes',                   ['11c'],        ['Diabetic', 'Diabetes']),
    ('1030', 'Diagnostic/Lab/Radiology',   ['3','8a2','8b2'], ['Diagnostic', 'X-Ray', 'Lab Services', 'Radiology']),
    ('1050', 'Fitness',                    ['13b','14c4'], ['Fitness']),
    ('1060', 'Meals',                      ['13c'],        ['Meals']),
    ('1200', 'Kidney',                     ['12'],         ['Kidney', 'Renal', 'Dialysis']),
    ('1300', 'Preventive Dental',          ['16a'],        ['Preventive Dental', 'Medicare Dental']),
    ('1301', 'Comprehensive Dental',       ['16b','16c1','16c'], ['Comprehensive Dental', 'Restorative']),
    ('1400', 'Hearing',                    ['18a','18b','18b1'], ['Hearing Exam', 'Hearing Aid', 'Fitting/Evaluation']),
    ('1500', 'Vision',                     ['17a','17a1','17b','17b2','17b3'], ['Eye Exam', 'Routine Eye', 'Eyewear', 'Eyeglasses', 'Contact Lenses']),
    ('1610', 'Prosthetics',                ['11b'],        ['Prosthetic']),
    ('1700', 'Preventive',                 ['14a','14b','14c'], ['Preventive', 'Wellness Visit']),
    ('1800', 'Transportation',             ['15'],         ['Transportation']),
    ('1900', 'Acupuncture',                ['13a'],        ['Acupuncture']),
    ('2100', 'OTC',                        ['13b','13e'],  ['Over-the-Counter', 'OTC']),
]

def row_matches_target(row, codes, keywords):
    cat = (row.get('category') or '').lower()
    for c in codes:
        if f'({c})' in cat or f'({c}1)' in cat or f'({c}2)' in cat:
            return True
    for kw in keywords:
        if kw.lower() in cat:
            return True
    return False

out_df = pd.DataFrame(output_rows)
if not out_df.empty:
    out_df['benefitid'] = out_df['benefitid'].astype(str)
    plans_with_bid = out_df.groupby('benefitid')['planid'].nunique().to_dict()
    n_plans = out_df['planid'].nunique()
else:
    plans_with_bid = {}
    n_plans = 0

coverage = []
for bid, name, codes, keywords in BENEFIT_TARGETS:
    plans_with_data = int(plans_with_bid.get(bid, 0))
    if plans_with_data > 0:
        verdict = 'OK'
    else:
        matched = sum(1 for r in clean if row_matches_target(r, codes, keywords))
        verdict = 'DATA_SHAPE' if matched == 0 else 'LLM_FAILED'
    coverage.append({
        'benefitid': bid,
        'name': name,
        'plans_w_data': plans_with_data,
        'pct_plans': f'{100*plans_with_data/max(n_plans,1):.0f}%',
        'verdict': verdict,
    })

cov_df = pd.DataFrame(coverage)
n_ok        = (cov_df['verdict']=='OK').sum()
n_dataShape = (cov_df['verdict']=='DATA_SHAPE').sum()
n_llmFailed = (cov_df['verdict']=='LLM_FAILED').sum()
n_total     = len(cov_df)

print(f'Coverage: {n_ok}/{n_total} OK | {n_dataShape} DATA_SHAPE | {n_llmFailed} LLM_FAILED\n')
display(cov_df.sort_values(['verdict', 'plans_w_data'], ascending=[True, False]))


## 5. Bottom-line verdict

In [ ]:
print('=' * 70)
print(f'Run: {RUN_LABEL}    Total time: {total_time:.1f}s')
print('=' * 70)
print(f'Server build: {server_ver}')
print(f'Plans: {output_data.get("plan_count")} processed | {output_data.get("result_count")} rows out')
print(f'Coverage: {n_ok}/{n_total} OK | {n_dataShape} data-shape | {n_llmFailed} LLM-failed')
print()
print('NEXT ACTION:')
if n_dataShape > n_llmFailed and n_dataShape > 2:
    failing = cov_df[cov_df['verdict']=='DATA_SHAPE']['name'].tolist()
    print(f'  → BENEFIT_TARGETS issue. Update keywords/codes for: {", ".join(failing)}')
elif n_llmFailed > 0:
    failing = cov_df[cov_df['verdict']=='LLM_FAILED']['name'].tolist()
    print(f'  → LLM issue. These benefits found input rows but returned 0 output:')
    print(f'    {", ".join(failing)}')
    print(f'    Either split into sub-targets or add focused few-shot examples.')
else:
    print(f'  ✓ All benefits extracting. Ready to validate quality against reference.')


## 6. Scoreboard append

In [ ]:
scoreboard_row = {
    'timestamp':       datetime.now().isoformat(timespec='seconds'),
    'run_label':       RUN_LABEL,
    'server_ver':      server_ver,
    'load_id':         load_id,
    'input_plans':     len(input_plans),
    'output_plans':    output_data.get('plan_count'),
    'output_rows':     output_data.get('result_count'),
    'wall_time_s':     round(total_time, 1),
    'benefits_ok':     n_ok,
    'benefits_data_shape': n_dataShape,
    'benefits_llm_failed': n_llmFailed,
}
sb_df = pd.DataFrame([scoreboard_row])
if SCOREBOARD_PATH.exists():
    existing = pd.read_csv(SCOREBOARD_PATH)
    sb_df = pd.concat([existing, sb_df], ignore_index=True)
sb_df.to_csv(SCOREBOARD_PATH, index=False)
print(f'Appended to {SCOREBOARD_PATH}')
display(sb_df.tail(10))
